![DataProjectLab](../../logo_dataprojectlab.png)


# Notebook 3 — Dashboard Power BI & Storytelling
**SOLUTION COMPLETE**

**Projet :** HotelChain West Analytics  
**Prerequis :** Notebook 2 termine, vues et procedure stockee creees dans HotelChainDB  
**Auteur :** DataProjectLab


---
## Contexte

Les analyses SQL des Notebooks 1 et 2 ont produit des reponses precises aux questions business d'HotelChain West. Ce Notebook 3 transforme ces analyses en un **outil de pilotage visuel 5 pages** accessible a la direction sans connaissance SQL.

**Ce que ce notebook couvre :**
1. Connexion Power BI → SQL Server (Import Mode)
2. Construction du modele de donnees en etoile
3. Les 12 mesures DAX avec valeurs de validation
4. Architecture page par page
5. Checklist de validation finale

---
## Etape 1 — Connexion Power BI → SQL Server

### METHODE — Import vs DirectQuery

Power BI propose deux modes de connexion a SQL Server :

| | Import | DirectQuery |
|---|---|---|
| Donnees | Copiees dans Power BI | Restent dans SQL Server |
| Vitesse | Rapide (tout en memoire) | Lent (requete SQL a chaque clic) |
| Fraicheur | Refresh manuel ou programme | Toujours a jour |
| DAX | Toutes les mesures disponibles | Certaines mesures limitees |
| Taille max | ~1 Go compresse | Illimite |

**Pour ce projet : Import.** Les donnees historiques (2022-2024) ne changent pas en temps reel. L'Import offre la meilleure experience utilisateur et donne acces a toutes les fonctionnalites DAX.

**Procedure de connexion :**
```
1. Power BI Desktop → Accueil → Obtenir des donnees → SQL Server
2. Serveur   : localhost
3. Base      : HotelChainDB
4. Mode      : Import
5. Selectonner les VUES (pas les tables brutes) :
     vw_clients_propres
     vw_reservations_propres
     vw_paiements_valides
6. Selectonner aussi les tables de reference :
     hotels
     chambres
     services
7. Cliquer Charger
```

> **Pourquoi charger les vues et non les tables brutes ?**
> Les vues `vw_reservations_propres` et `vw_paiements_valides` excluent deja les anomalies corrigees dans le NB2. Si on charge les tables brutes, les 5 montants negatifs et les 3 nb_nuits=0 apparaitront dans le dashboard.

In [ ]:
-- Verifier dans SSMS que les 3 vues existent avant de connecter Power BI
SELECT
    TABLE_NAME  AS objet,
    TABLE_TYPE  AS type
FROM INFORMATION_SCHEMA.TABLES
WHERE TABLE_SCHEMA = 'dbo'
ORDER BY TABLE_TYPE DESC, TABLE_NAME;

-- Resultat attendu :
-- hotels                    BASE TABLE
-- chambres                  BASE TABLE
-- clients                   BASE TABLE
-- reservations              BASE TABLE
-- paiements                 BASE TABLE
-- services                  BASE TABLE
-- vw_clients_propres        VIEW
-- vw_paiements_valides      VIEW
-- vw_reservations_propres   VIEW


---
## Etape 2 — Modele de donnees en etoile

### METHODE — Pourquoi le schema en etoile ?

Power BI est optimise pour le schema en etoile : une **table de faits** centrale (les transactions) entouree de **tables de dimension** (les attributs descriptifs).

**Avantages vs schema en flocon ou tables brutes jointurees :**
- Calculs DAX plus rapides (moteur VertiPaq optimise pour ce schema)
- Filtres et slicers plus intuitifs
- Moins de relations ambigues

```
         hotels (dimension)      Calendrier (dimension temps)
              |                        |
              | hotel_id               | date
              |                        |
chambres ─────┤                        |
  chambre_id  |                        |
              |                        |
vw_clients ───┤── vw_reservations (FAIT CENTRAL) ──── vw_paiements
  client_id   |       reservation_id                  reservation_id
              |           |
              |       services (fait secondaire)
              |       reservation_id
```

### Relations a creer dans Power BI (vue Modele)

| De | Vers | Cle | Cardinalite | Direction filtre |
|---|---|---|---|---|
| hotels[hotel_id] | vw_reservations_propres[hotel_id] | hotel_id | 1:N | Single (→) |
| chambres[chambre_id] | vw_reservations_propres[chambre_id] | chambre_id | 1:N | Single (→) |
| vw_clients_propres[client_id] | vw_reservations_propres[client_id] | client_id | 1:N | Single (→) |
| vw_reservations_propres[reservation_id] | vw_paiements_valides[reservation_id] | reservation_id | 1:N | Single (→) |
| vw_reservations_propres[reservation_id] | services[reservation_id] | reservation_id | 1:N | Single (→) |
| Calendrier[Date] | vw_reservations_propres[date_arrivee] | date | 1:N | Single (→) |

> **Regle d'or :** toujours Single direction (→). La direction Both (↔) peut creer des ambiguites et ralentir les calculs. Si un visuel ne filtre pas comme attendu, verifier d'abord la direction des relations.

In [ ]:
-- Formule DAX : Table Calendrier (a creer dans Power BI)
-- Accueil → Nouvelle table → coller cette formule

Calendrier =
ADDCOLUMNS(
    CALENDAR(DATE(2022,1,1), DATE(2024,12,31)),
    "Annee",        YEAR([Date]),
    "Mois_Num",     MONTH([Date]),
    "Mois_Nom",     FORMAT([Date], "MMM"),
    "Mois_Court",   FORMAT([Date], "MMM YYYY"),
    "Trimestre",    "T" & QUARTER([Date]),
    "Semestre",     IF(MONTH([Date]) <= 6, "S1", "S2"),
    "Semaine",      WEEKNUM([Date]),
    "Jour_Semaine", FORMAT([Date], "ddd"),
    "Num_Jour",     WEEKDAY([Date], 2),
    "Est_Weekend",  IF(WEEKDAY([Date], 2) >= 6, 1, 0),
    "Saison",       IF(
                        MONTH([Date]) IN {12, 1, 2, 7, 8},
                        "Haute saison",
                        "Basse saison"
                    )
)

-- Apres creation :
-- Clic droit sur la table Calendrier → Marquer comme table de dates
-- Choisir la colonne Date
-- Cette etape est OBLIGATOIRE pour que PREVIOUSMONTH() et SAMEPERIODLASTYEAR() fonctionnent


### INTERPRETATION

La table Calendrier est la **colonne vertebrale temporelle** du dashboard. Sans elle, les mesures DAX de Time Intelligence (`PREVIOUSMONTH`, `SAMEPERIODLASTYEAR`, `DATESYTD`) ne fonctionnent pas.

Le champ `Saison` (Haute/Basse) sera utilise comme slicer sur la Page 2 pour comparer les performances saisonnieres directement dans Power BI, sans recalcul SQL.

> **Etape critique souvent oubliee :** apres avoir cree la table, il faut imperativement la **marquer comme table de dates** dans Power BI. Sans ca, `PREVIOUSMONTH()` retourne une erreur silencieuse ou un resultat vide.

---
## Etape 3 — Les 12 mesures DAX

### METHODE — La table _Mesures

En Power BI, la bonne pratique est de creer une table vide nommee `_Mesures` qui contiendra toutes les mesures. Avantages :
- Toutes les mesures regroupees dans le meme dossier du panneau Champs
- Plus facile a maintenir et a partager
- Le `_` au debut la place en haut de la liste alphabetique

```
Pour creer la table _Mesures :
Accueil → Entrer des donnees → Nommer '_Mesures' → Charger
Puis supprimer la colonne 'Colonne1' automatiquement ajoutee
```

In [ ]:
-- ════════════════════════════════════════════
-- MESURES DE BASE
-- ════════════════════════════════════════════

-- Mesure 1 : Total reservations
Total Reservations =
COUNTROWS(vw_reservations_propres)
-- Valeur attendue sans filtre : 7 992

-- Mesure 2 : Revenu total chambres
Revenu Total =
CALCULATE(
    SUM(vw_paiements_valides[montant]),
    vw_paiements_valides[statut_paiement] = "Valide"
)
-- Valeur attendue sans filtre : 2 874 928 662 FCFA

-- Mesure 3 : CSAT moyen (sur sejours termines uniquement)
CSAT Moyen =
CALCULATE(
    AVERAGE(vw_reservations_propres[csat]),
    vw_reservations_propres[statut] = "Terminee"
)
-- Valeur attendue sans filtre : 4.01

-- Mesure 4 : Taux annulation
Taux Annulation =
DIVIDE(
    CALCULATE(
        COUNTROWS(vw_reservations_propres),
        vw_reservations_propres[statut] = "Annulee"
    ),
    COUNTROWS(reservations)   -- table brute pour avoir le total reel
)
-- Valeur attendue sans filtre : 3.8%

-- Mesure 5 : Nuits vendues
Nuits Vendues =
CALCULATE(
    SUM(vw_reservations_propres[nb_nuits]),
    vw_reservations_propres[statut] IN {"Terminee", "En cours"}
)
-- Valeur attendue sans filtre : 19 978


### INTERPRETATION — Mesures de base

Ces 5 mesures sont les **KPIs de la Page 1 (Vue executive)**. Elles seront affichees sous forme de cartes KPI avec couleur conditionnelle.

**Point technique important sur le CSAT :**
La colonne `csat` est stockee en `VARCHAR(5)` dans SQL Server (cf. NB1). Power BI la charge comme texte. Il faut la convertir lors du chargement :
```
Power Query → vw_reservations_propres → colonne csat
→ Transformer → Type → Nombre entier
→ Remplacer les erreurs de conversion par null
```
Sans cette conversion, `AVERAGE(csat)` retourne une erreur.

In [ ]:
-- ════════════════════════════════════════════
-- MESURES AVANCEES
-- ════════════════════════════════════════════

-- Mesure 6 : Taux d'occupation
-- Formule : nuits vendues / (nb chambres x nb jours periode selectionnee)
Taux Occupation =
VAR _nuits_vendues = [Nuits Vendues]
VAR _capacite =
    SUMX(
        hotels,
        hotels[nb_chambres] *
        COUNTROWS(DISTINCT(Calendrier[Date]))
    )
RETURN
DIVIDE(_nuits_vendues, _capacite)
-- Valeur attendue sans filtre sur toute la periode : ~4.4% (moyenne ponderee)

-- Mesure 7 : ADR (Average Daily Rate)
ADR =
DIVIDE([Revenu Total], [Nuits Vendues])
-- Valeur attendue sans filtre : ~143 900 FCFA (moyenne tous hotels)

-- Mesure 8 : RevPAR
RevPAR =
[ADR] * [Taux Occupation]
-- Valeur attendue sans filtre : ~6 300 FCFA (moyenne tous hotels)

-- Mesure 9 : Revenu extras (services)
Revenu Extras =
SUM(services[montant])
-- Valeur attendue sans filtre : 278 143 196 FCFA

-- Mesure 10 : Revenu total consolide (chambres + extras)
Revenu Total Consolide =
[Revenu Total] + [Revenu Extras]
-- Valeur attendue sans filtre : 3 153 071 858 FCFA (~3.15 Mrd FCFA)


In [ ]:
-- ════════════════════════════════════════════
-- MESURES TEMPORELLES
-- ════════════════════════════════════════════

-- Mesure 11 : Revenu mois precedent (Time Intelligence)
Revenu Mois Prec =
CALCULATE(
    [Revenu Total],
    PREVIOUSMONTH(Calendrier[Date])
)
-- Necessite que Calendrier soit marque comme table de dates

-- Mesure 12 : Variation revenu vs mois precedent
Variation Revenu =
VAR _actuel = [Revenu Total]
VAR _prec   = [Revenu Mois Prec]
RETURN
DIVIDE(_actuel - _prec, _prec)
-- Afficher en format pourcentage avec fleche conditionnelle :
-- Si > 0 : fleche verte ▲
-- Si < 0 : fleche rouge ▼
-- Si BLANK : tiret —


### INTERPRETATION — Mesures avancees et temporelles

**Sur le Taux d'Occupation :**
La mesure utilise `SUMX(hotels, nb_chambres * COUNTROWS(DISTINCT(Calendrier[Date])))` pour calculer la capacite theorique dynamiquement selon la periode selectionnee. Si l'utilisateur filtre sur 'Janvier 2023', la capacite se recalcule automatiquement sur 31 jours x nb chambres. C'est plus robuste qu'une constante codee en dur.

**Sur la Variation Revenu :**
L'utilisation de `VAR` (variables DAX) ameliore la lisibilite et les performances. Une mesure avec 2 references a `[Revenu Total]` sans VAR calculerait la mesure deux fois. Avec VAR, elle est calculee une seule fois et stockee en memoire.

**Valeurs de validation sans filtre :**

| Mesure | Valeur attendue |
|---|---|
| [Total Reservations] | **7 992** |
| [Revenu Total] | **2 874 928 662 FCFA** |
| [CSAT Moyen] | **4.01** |
| [Taux Annulation] | **3.8%** |
| [Nuits Vendues] | **19 978** |
| [Revenu Extras] | **278 143 196 FCFA** |
| [Revenu Total Consolide] | **3 153 071 858 FCFA** |

> Si une valeur ne correspond pas : verifier en priorite (1) la direction des relations, (2) que la table Calendrier est marquee comme table de dates, (3) que la colonne `csat` a ete convertie en entier dans Power Query.

---
## Etape 4 — Architecture page par page

### Page 1 — Vue executive

**Objectif :** donner a la direction une vue globale en 30 secondes.

**Visuels a construire :**

**1. Bandeau de 4 cartes KPI** (haut de page)
```
[Revenu Total]        [CSAT Moyen]       [Taux Annulation]    [Nuits Vendues]
2 874 928 662 FCFA    4.01 / 5           3.8%                 19 978
```
- Couleur : vert si KPI favorable, rouge si defavorable
- Sous-titre : variation vs periode precedente avec [Variation Revenu]

**2. Courbe : evolution mensuelle du RevPAR par hotel (2022-2024)**
- X : Calendrier[Mois_Court]
- Y : [RevPAR]
- Legende : hotels[nom]
- Ligne de reference : RevPAR moyen global (~6 300 FCFA)

**3. Carte geographique : revenu par pays**
- Utiliser hotels[pays] comme emplacement
- Taille des bulles : [Revenu Total]
- Pays couverts : Cote d'Ivoire, Senegal, Cameroun, Ghana

**4. Barres horizontales : top 5 canaux par revenu**
- Y : vw_reservations_propres[canal]
- X : [Revenu Total]
- Valeurs attendues :
  - Direct : 881 M FCFA
  - Booking.com : 808 M FCFA
  - Agence voyage : 416 M FCFA
  - Expedia : 403 M FCFA
  - Corporate : 265 M FCFA

**5. Slicers** : Calendrier[Annee], hotels[nom], Calendrier[Saison]

In [ ]:
-- Requete de validation Page 1 (executer dans SSMS)
SELECT
    YEAR(p.date_paiement)              AS annee,
    ROUND(SUM(p.montant) / 1000000, 1) AS revenu_millions
FROM vw_paiements_valides p
GROUP BY YEAR(p.date_paiement)
ORDER BY annee;

-- Resultat attendu (a comparer avec la courbe Power BI) :
-- 2022 : ~877 M FCFA
-- 2023 : ~1 178 M FCFA
-- 2024 (6 mois) : ~820 M FCFA


### Page 2 — Occupation & Saisonnalite

**Objectif :** identifier les periodes de haute et basse saison par hotel.

**1. Matrice (heatmap) : taux d'occupation par hotel x mois**
- Lignes : hotels[nom]
- Colonnes : Calendrier[Mois_Num]
- Valeurs : [Taux Occupation]
- Mise en forme conditionnelle par regles :
  - < 3% : rouge
  - 3% a 6% : orange
  - > 6% : vert

Valeurs 2023 attendues dans la matrice :

| Hotel | Jan | Fev | Mar | Avr | Mai | Jun | Jul | Aou | Sep | Oct | Nov | Dec |
|---|---|---|---|---|---|---|---|---|---|---|---|---|
| Douala | 6.9 | 7.7 | 6.8 | 5.8 | 8.2 | 5.6 | 8.0 | **8.5** | 5.5 | 6.5 | 6.0 | 6.2 |
| Cocody | 6.8 | 7.0 | 5.0 | 4.9 | 4.6 | 3.6 | 6.2 | **7.3** | 4.1 | 5.2 | 5.0 | 5.8 |
| Dakar  | 5.4 | 3.7 | 4.5 | 4.6 | 4.6 | 3.7 | **6.9** | 5.8 | 4.6 | 3.9 | 4.1 | 5.5 |
| Plateau| 6.4 | 4.3 | 3.4 | 3.7 | 3.4 | 3.6 | 4.9 | 4.5 | 3.3 | 4.0 | 3.6 | 3.5 |
| Accra  | 3.5 | 2.4 | **2.1** | 3.6 | 2.7 | 2.9 | 2.7 | 3.7 | 3.1 | 2.8 | 3.1 | 3.1 |

**2. Courbe : taux d'occupation mensuel par hotel**
- Ligne de cible a 6% (objectif groupe)

**3. Barres groupees : haute saison vs basse saison**
- Slicer Calendrier[Saison] active le filtre automatiquement

**4. Carte KPI** : meilleur mois (Douala aout 2023 : 8.5%), pire mois (Accra mars 2023 : 2.1%)

In [ ]:
-- Requete de validation Page 2 (executer dans SSMS)
SELECT
    h.nom,
    YEAR(r.date_arrivee)  AS annee,
    MONTH(r.date_arrivee) AS mois,
    ROUND(
        SUM(r.nb_nuits) * 100.0
        / (h.nb_chambres * DAY(EOMONTH(DATEFROMPARTS(
            YEAR(r.date_arrivee), MONTH(r.date_arrivee), 1))))
    , 1)                  AS taux_occupation
FROM vw_reservations_propres r
JOIN hotels h ON r.hotel_id = h.hotel_id
WHERE r.statut IN ('Terminee', 'En cours')
  AND YEAR(r.date_arrivee) = 2023
GROUP BY h.nom, h.hotel_id, h.nb_chambres,
         YEAR(r.date_arrivee), MONTH(r.date_arrivee)
ORDER BY h.nom, mois;

-- Ces chiffres doivent correspondre exactement a la heatmap Power BI
-- Si ecart : verifier que la relation Calendrier → vw_reservations est sur date_arrivee


### Page 3 — Revenus & Tarification

**Objectif :** analyser la structure des revenus et identifier les leviers tarifaires.

**1. Aires empilees : revenus mensuels par canal**
- X : Calendrier[Mois_Court]
- Y : [Revenu Total]
- Legende : vw_reservations_propres[canal]

**2. Scatter : ADR x Taux occupation par hotel**
- X : [Taux Occupation]
- Y : [ADR]
- Taille bulle : [Revenu Total]
- Legende : hotels[nom]
- Valeurs attendues :
  - Douala : (6.40%, 105 248 FCFA) → bulle moyenne
  - Cocody : (5.53%, 96 100 FCFA)
  - Dakar  : (4.49%, 113 606 FCFA)
  - Plateau: (3.82%, 114 083 FCFA)
  - Accra  : (2.77%, 142 970 FCFA) → bulle plus grande (revenu total plus eleve)

> Ce scatter est le visuel le plus impactant du dashboard. Il montre immediatement qu'Accra est hors tendance : prix le plus eleve mais occupation la plus basse. La zone 'sweet spot' se situe entre 5-7% d'occupation et 100-130k FCFA ADR.

**3. Tableau : synthese RevPAR par hotel et par trimestre**
- Lignes : hotels[nom]
- Colonnes : Calendrier[Trimestre]
- Valeurs : [RevPAR]

**4. Barres : ADR par type de chambre**
- Valeurs attendues (ADR reel) :
  - Suite : 294 491 FCFA
  - Deluxe : 149 571 FCFA
  - Superieure : 100 043 FCFA
  - Standard : 66 284 FCFA

### Page 4 — Clients & Satisfaction

**Objectif :** comprendre qui sont les clients et ou agir sur la satisfaction.

**1. Tableau de segmentation clients**
- Source : requete synthese CLV (NB2 Etape 6, Requete 2/2)
- Colonnes : quartile, nb_clients, revenu_moyen, sejours_moyens, pct_revenu_total
- Mise en forme conditionnelle sur pct_revenu_total

**2. Barres : CSAT moyen par hotel avec cible 4.0**
- Valeurs attendues :

| Hotel | CSAT | vs cible 4.0 |
|---|---|---|
| HotelChain Cocody | **4.05** | +0.05 ✓ |
| HotelChain Dakar | 4.03 | +0.03 ✓ |
| HotelChain Douala | 4.01 | +0.01 ✓ |
| HotelChain Plateau | 4.00 | 0.00 = |
| HotelChain Accra | **3.98** | -0.02 ✗ |

> Accra est le seul hotel sous la cible de 4.0. Combine avec son RevPAR le plus bas, c'est le signal le plus fort du dashboard : Accra cumule les problemes de remplissage ET de satisfaction.

**3. Anneau : repartition des nationalites (top 5)**
- Valeurs attendues :
  - Ivoirien : 902 clients (30.1%)
  - Senegalais : 454 (15.1%)
  - Camerounais : 380 (12.7%)
  - Ghaneeen : 301 (10.0%)
  - Autre : 275 (9.2%)

**4. Carte KPI double** : % clients fideles (25%) vs nouveaux (75%)

### Page 5 — Services & Extras

**Objectif :** mesurer la contribution des revenus extras et identifier les leviers d'upselling.

**1. Barres horizontales : revenu extras par categorie**
- Valeurs attendues :

| Categorie | Revenu | Ticket moyen | Nb transactions |
|---|---|---|---|
| Restaurant | 68 932 931 FCFA | 24 707 FCFA | 2 790 |
| Salle conference | 65 059 591 FCFA | **132 504 FCFA** | 491 |
| Spa & Bien-etre | 52 611 604 FCFA | 74 839 FCFA | 703 |
| Room service | 33 511 751 FCFA | 19 552 FCFA | 1 714 |
| Transport | 29 075 461 FCFA | 32 669 FCFA | 890 |
| Minibar | 19 514 727 FCFA | 13 321 FCFA | 1 465 |
| Blanchisserie | 9 437 131 FCFA | 9 955 FCFA | 948 |

**2. Jauge : ratio extras / revenu total**
- Valeur actuelle : **9.7%** (278 M / 2 875 M)
- Cible : 15% (norme industrie 4-5 etoiles)
- Interpretation : HotelChain West sous-exploite ses services annexes

**3. Courbe : evolution mensuelle du revenu extras**
- Superposer avec le revenu chambre pour montrer la correlation

**4. Carte KPI** : Revenu consolide = [Revenu Total Consolide] = **3 153 071 858 FCFA**

In [ ]:
-- Requete de validation Page 5 (executer dans SSMS)
SELECT
    s.categorie,
    COUNT(*)                           AS nb_transactions,
    ROUND(SUM(s.montant), 0)           AS revenu_total,
    ROUND(AVG(s.montant), 0)           AS ticket_moyen,
    ROUND(SUM(s.montant) * 100.0
          / (SELECT SUM(montant) FROM services), 1) AS part_pct
FROM services s
GROUP BY s.categorie
ORDER BY revenu_total DESC;

-- Ratio extras / revenu chambre
SELECT
    ROUND(
        (SELECT SUM(montant) FROM services) * 100.0
        / (SELECT SUM(montant) FROM vw_paiements_valides)
    , 1) AS ratio_extras_pct;

-- Resultat attendu : 9.7%
-- Ecart avec norme industrie : -5.3 points (cible 15%)
-- Potentiel de revenu additionnel si 15% atteint :
-- 2 874 928 662 x 0.15 = 431 239 299 FCFA vs 278 143 196 FCFA actuels
-- Ecart = +153 096 103 FCFA de revenu extras a aller chercher


---
## Etape 5 — Sous-titres dynamiques et slicers

### METHODE

Les sous-titres dynamiques rendent le dashboard professionnel : quand un slicer est active, le sous-titre se met a jour automatiquement. L'utilisateur sait toujours sur quel perimetre il travaille.

`SELECTEDVALUE(table[colonne], valeur_par_defaut)` retourne la valeur selectionnee dans un slicer. Si rien n'est selectionne (ou plusieurs valeurs), il retourne la valeur par defaut.

In [ ]:
-- Sous-titres dynamiques pour chaque page

Sous Titre Page 1 =
VAR _hotel  = SELECTEDVALUE(hotels[nom],        "Tous les hotels")
VAR _annee  = SELECTEDVALUE(Calendrier[Annee],  "2022 - 2024")
VAR _saison = SELECTEDVALUE(Calendrier[Saison], "Toutes saisons")
RETURN
"HotelChain West  ·  " & _hotel & "  ·  " & _annee & "  ·  " & _saison

Sous Titre Page 2 =
VAR _taux  = FORMAT([Taux Occupation], "0.0%")
VAR _nuits = FORMAT([Nuits Vendues],   "#,##0")
RETURN
_nuits & " nuits vendues  ·  Taux occupation moyen : " & _taux

Sous Titre Page 3 =
VAR _canal = SELECTEDVALUE(vw_reservations_propres[canal], "Tous canaux")
VAR _adr   = FORMAT([ADR], "#,##0") & " FCFA"
RETURN
"Canal : " & _canal & "  ·  ADR moyen : " & _adr

Sous Titre Page 5 =
VAR _pct = FORMAT(DIVIDE([Revenu Extras], [Revenu Total Consolide]), "0.0%")
RETURN
"Extras : " & _pct & " du revenu consolide  ·  Cible : 15%"


---
## Etape 6 — Checklist de validation finale

### METHODE

Avant de livrer un dashboard, toujours valider chaque mesure en la comparant avec le resultat SQL de reference. Une mesure DAX incorrecte peut passer inapercue pendant des mois si personne ne la valide contre la source.

**Procedure :** ouvrir SSMS et Power BI cote a cote, enlever tous les filtres dans Power BI, et comparer chaque valeur.

| Mesure DAX | Valeur attendue (sans filtre) | Statut |
|---|---|---|
| [Total Reservations] | **7 992** | [ ] |
| [Revenu Total] | **2 874 928 662 FCFA** | [ ] |
| [CSAT Moyen] | **4.01** | [ ] |
| [Taux Annulation] | **3.8%** | [ ] |
| [Nuits Vendues] | **19 978** | [ ] |
| [Revenu Extras] | **278 143 196 FCFA** | [ ] |
| [Revenu Total Consolide] | **3 153 071 858 FCFA** | [ ] |
| [ADR] filtre Douala | **105 248 FCFA** | [ ] |
| [RevPAR] filtre Douala | **~8 151 FCFA** | [ ] |
| [CSAT Moyen] filtre Cocody | **4.05** | [ ] |

### Erreurs les plus frequentes et solutions

| Erreur | Cause probable | Solution |
|---|---|---|
| CSAT retourne BLANK | csat non converti en entier | Power Query → colonne csat → Type Entier |
| PREVIOUSMONTH retourne BLANK | Calendrier non marque table de dates | Clic droit Calendrier → Marquer comme table de dates |
| Taux Occupation trop eleve | Relation Calendrier mal configuree | Verifier la relation sur date_arrivee |
| Extras ne filtrent pas par hotel | Relation services → hotels manquante | Ajouter la relation services[hotel_id] → hotels[hotel_id] |
| Revenu double | Relation Both sur paiements | Passer en Single direction |


In [ ]:
-- Requete SQL finale de validation globale (executer dans SSMS)
-- Ces chiffres doivent correspondre exactement aux cartes KPI de la Page 1

SELECT
    COUNT(DISTINCT r.reservation_id)               AS total_reservations,
    ROUND(SUM(p.montant) / 1000000000.0, 3)        AS revenu_mrd_fcfa,
    ROUND(AVG(TRY_CAST(r.csat AS FLOAT)), 2)        AS csat_moyen,
    ROUND(
        SUM(CASE WHEN res.statut = 'Annulee'
                 THEN 1 ELSE 0 END) * 100.0
        / COUNT(res.reservation_id)
    , 1)                                           AS taux_annulation_pct,
    SUM(CASE WHEN r.statut IN ('Terminee','En cours')
             THEN r.nb_nuits ELSE 0 END)           AS nuits_vendues,
    (SELECT ROUND(SUM(montant) / 1000000.0, 1)
     FROM services)                                AS extras_millions
FROM vw_reservations_propres r
LEFT JOIN vw_paiements_valides p
    ON r.reservation_id = p.reservation_id
CROSS JOIN (SELECT COUNT(*) AS reservation_id FROM reservations) res;

-- Resultats attendus :
-- total_reservations : 7 992
-- revenu_mrd_fcfa    : 2.875
-- csat_moyen         : 4.01
-- taux_annulation    : 3.8%
-- nuits_vendues      : 19 978
-- extras_millions    : 278.1


---
## Bilan du Notebook 3

### Ce que le dashboard HotelChain West permet de piloter

| Page | Question business repondue |
|---|---|
| 1 — Vue executive | Quel est le revenu global ? Quelle evolution ? |
| 2 — Occupation | Quand et ou les hotels sont-ils pleins ou vides ? |
| 3 — Revenus | Quels canaux et types de chambres sont les plus rentables ? |
| 4 — Clients | Qui sont les clients VIP ? Ou agir sur la satisfaction ? |
| 5 — Extras | Quel est le potentiel d'upselling non exploite ? |

### Les 3 recommandations prioritaires issues du dashboard

**1. Accra : intervention urgente** — seul hotel sous la cible CSAT (3.98) ET dernier au RevPAR (4 851 FCFA). Baisser l'ADR de 10-15% et lancer une campagne de remplissage pour rattraper Douala.

**2. Canal direct : investir** — le canal Direct genere 881 M FCFA avec 0% de commission. Chaque point de part de marche gagne sur les OTA recupere ~9 M FCFA de marge nette. ROI positif sur un site web optimise.

**3. Services extras : doubler l'objectif** — 9.7% du revenu vs norme 15%. Potentiel additionnel de **+153 M FCFA** par an en atteignant la norme. Priorite : la Salle conference (ticket moyen 132 504 FCFA, 7x plus rentable que le restaurant).

---

**DataProjectLab** — apprendre la data sur des cas concrets, structures et orientes metier.